# Real-Time Face Verification with a Siamese Neural Network

An experimental biometric-verification pipeline that compares a live webcam image with an enrolled user's reference images. This is **one-to-one verification**, not multi-person face recognition.

## Project snapshot

| | |
|---|---|
| **Goal** | Decide whether a live capture matches one enrolled identity |
| **Data** | Locally captured anchor/positive images plus LFW images as non-matches |
| **Approach** | Shared CNN embeddings, L1 distance, sigmoid match score, OpenCV webcam loop |
| **Evaluation** | Prototype batch precision/recall at a `0.5` threshold; production evaluation is not implemented |
| **Status** | End-to-end experimental workflow implemented; no execution results are saved in this notebook |

## Run requirements

Open the notebook from its project folder, or from a Jupyter server launched anywhere inside this repository. A webcam, OpenCV GUI support, TensorFlow, KaggleHub access, and sufficient storage for LFW and captured images are required. Training the custom CNN is compute-intensive; a GPU is recommended.

> **Scope and privacy:** This is a learning prototype, not a production biometric or security system. Capture images only with informed consent, keep images/models private, and follow applicable biometric-privacy rules. Performance can vary with lighting, pose, cameras, and demographic groups.


## 1. Environment Setup

This section imports the dependencies used throughout the project and creates the folder structure that will hold the three core data categories:
- **anchor** images of the enrolled user
- **positive** images of the same user
- **negative** images of different people


In [ ]:
# Core libraries used across data collection, preprocessing, modelling, and visualisation.
import os
import random
import shutil
import sys
from pathlib import Path

# Resolve shared repository paths whether Jupyter starts at the repo root or a project folder.
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / 'portfolio_utils').is_dir():
        repo_root = candidate
        break
else:
    raise FileNotFoundError("Could not locate the repository root containing 'portfolio_utils'.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from portfolio_utils import ARTIFACTS_DIR, DATASETS_DIR

import cv2
import numpy as np
from matplotlib import pyplot as plt


In [ ]:
# TensorFlow / Keras components used to build the Siamese network.
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, Conv2D, Dense, MaxPooling2D, Flatten, Input
import tensorflow as tf


In [ ]:
# Define the working directories for the three image groups used in training:
# - anchor: reference images of the enrolled person
# - positive: additional images of the same person
# - negative: images of other people (used as non-matches)
FACE_DATA_DIR = DATASETS_DIR / 'facial-recognition'
POS_PATH = str(FACE_DATA_DIR / 'positive')
NEG_PATH = str(FACE_DATA_DIR / 'negative')
ANC_PATH = str(FACE_DATA_DIR / 'anchor')


In [ ]:
# Create the required directory structure if it does not already exist.
if not os.path.exists(POS_PATH):
    os.makedirs(POS_PATH)
if not os.path.exists(NEG_PATH):
    os.makedirs(NEG_PATH)
if not os.path.exists(ANC_PATH):
    os.makedirs(ANC_PATH)


## 2. Preparing the Negative Image Dataset

To teach the model what a **non-match** looks like, I use the **Labeled Faces in the Wild (LFW)** dataset as a source of negative examples.  
The cells below download the dataset, navigate to the image directory, and copy face images into the ignored local `negative` folder without modifying the KaggleHub cache.


In [ ]:
import kagglehub

# Download the LFW dataset, which is later used to populate the negative class.
path = kagglehub.dataset_download("jessicali9530/lfw-dataset")

print("Path to dataset files:", path)


In [ ]:
# Point directly to the folder that contains the downloaded face images.
path = os.path.join(path, 'lfw-deepfunneled', 'lfw-deepfunneled')


In [ ]:
# Copy downloaded LFW images into the negative folder without modifying
# KaggleHub's shared cache.
for folder in os.listdir(path):
    folder_path = os.path.join(path, folder)
    if os.path.isdir(folder_path):
        for img in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img)
            if os.path.isfile(img_path) and img.endswith('.jpg'):
                new_img_path = os.path.join(NEG_PATH, img)
                if not os.path.exists(new_img_path):
                    shutil.copy2(img_path, new_img_path)


In [ ]:
# Quick sanity check: confirm how many negative images were collected.
num_images = len(os.listdir(NEG_PATH))
print(f"Number of images in the negative folder: {num_images}")


## 3. Collecting Anchor and Positive Samples

The next step is to create person-specific training data using a webcam.

### Collection strategy
- **Anchor images** act as the reference identity.
- **Positive images** are additional images of the same person under slightly different conditions.
- Keeping the crop region fixed helps reduce unnecessary variation in framing.

### Keyboard controls
- Press **`a`** to capture an anchor image
- Press **`p`** to capture a positive image
- Press **`q`** to quit collection


### Capture loop

In [ ]:
# uuid is used to generate unique image filenames during webcam capture.
import uuid


In [ ]:
# Open a webcam session for collecting anchor and positive samples.
# Controls:
# - Press 'a' to save an anchor image
# - Press 'p' to save a positive image
# - Press 'q' to quit the collection loop
def center_square_crop(frame, max_size=600):
    """Return a centered square crop that works across camera resolutions."""
    height, width = frame.shape[:2]
    side = min(height, width, max_size)
    y_start = (height - side) // 2
    x_start = (width - side) // 2
    return frame[y_start:y_start + side, x_start:x_start + side]

cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Unable to read a frame from the webcam.")
        break

    # Use a centered square crop across different webcam resolutions.
    frame = center_square_crop(frame)

    # Capture anchor images and save them to the anchor folder.
    # Frames are resized to 250x250 before saving.
    if cv2.waitKey(1) & 0xFF == ord('a'):
        imgname = os.path.join(ANC_PATH, '{}.jpg'.format(uuid.uuid1()))
        cv2.imwrite(imgname, cv2.resize(frame, (250, 250)))

    # Capture positive images and save them to the positive folder.
    if cv2.waitKey(1) & 0xFF == ord('p'):
        imgname = os.path.join(POS_PATH, '{}.jpg'.format(uuid.uuid1()))
        cv2.imwrite(imgname, cv2.resize(frame, (250, 250)))

    # Display the live feed back to the user, flipped to behave like a mirror.
    cv2.imshow('Image Collection', cv2.flip(frame, 1))

    # Exit the loop cleanly when the user presses 'q'.
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the webcam and close all OpenCV windows.
cap.release()
cv2.destroyAllWindows()


## 4. Building the TensorFlow Data Pipeline

Once the images are collected, the project constructs paired datasets for Siamese learning.

### Pair labels
- **1** → anchor image paired with a positive image
- **0** → anchor image paired with a negative image

The preprocessing function standardises every image to a consistent size and pixel range before training.


In [ ]:
# Build file-path datasets for the three image groups.
# Each dataset is limited to 100 images for this experiment.
anchor = tf.data.Dataset.list_files(ANC_PATH + '/*.jpg').take(100)
positive = tf.data.Dataset.list_files(POS_PATH + '/*.jpg').take(100)
negative = tf.data.Dataset.list_files(NEG_PATH + '/*.jpg').take(100)


In [ ]:
# Inspect the TensorFlow dataset spec to verify what each element looks like.
print("Anchor dataset shape:", anchor.element_spec)


In [ ]:
# Preprocess an image file so it can be consumed by the network.
def preprocess(file_path):
    """Load, decode, resize, and normalise an image tensor."""
    byte_img = tf.io.read_file(file_path)
    img = tf.io.decode_jpeg(byte_img)
    img = tf.image.resize(img, (100, 100))
    img = img / 255.0
    return img


In [ ]:
# Create labelled pairs:
# - anchor + positive -> 1
# - anchor + negative -> 0
positives = tf.data.Dataset.zip((anchor, positive, tf.data.Dataset.from_tensor_slices(tf.ones(len(anchor)))))
negatives = tf.data.Dataset.zip((anchor, negative, tf.data.Dataset.from_tensor_slices(tf.zeros(len(anchor)))))
data = positives.concatenate(negatives)


In [ ]:
# Preview one raw sample before preprocessing.
sample = data.as_numpy_iterator()
print(sample.next())


In [ ]:
# Apply image preprocessing to both branches of the Siamese input pair.
def preprocess_twin(input_img, validation_img, label):
    """Preprocess anchor/validation image pairs while keeping the label unchanged."""
    return (preprocess(input_img), preprocess(validation_img), label)


In [ ]:
# Transform the dataset pipeline:
# 1. preprocess each image pair
# 2. cache for faster repeated access
# 3. shuffle once before creating non-overlapping pair-level splits
data = data.map(preprocess_twin)
data = data.cache()
data = data.shuffle(buffer_size=1024, seed=42, reshuffle_each_iteration=False)


In [ ]:
# Split the dataset into training and testing partitions, then batch/prefetch
# to improve throughput during model training and evaluation.
train_data = data.take(round(len(data)*.7))
train_data = train_data.batch(16).prefetch(8)

test_data = data.skip(round(len(data)*.7))
test_data = test_data.take(round(len(data)*.3))
test_data = test_data.batch(16).prefetch(8)


## 5. Defining the Siamese Network

This model is built in two parts:

### 5.1 Embedding network
A convolutional network converts each image into a dense numerical representation.

### 5.2 Similarity network
Two images are passed through the shared embedding network, then compared using an **L1 distance layer**.  
A final sigmoid classifier predicts whether the pair is a match.


In [ ]:
def make_embedding():
    """Build the embedding network used by both branches of the Siamese model."""
    inp = Input(shape=(100, 100, 3), name='input_image')

    # First convolutional block
    c1 = Conv2D(64, (10,10), activation='relu')(inp)
    m1 = MaxPooling2D((2,2), padding='same')(c1)

    # Second convolutional block
    c2 = Conv2D(128, (7,7), activation='relu')(m1)
    m2 = MaxPooling2D((2,2), padding='same')(c2)

    # Third convolutional block
    c3 = Conv2D(128, (4,4), activation='relu')(m2)
    m3 = MaxPooling2D((2,2), padding='same')(c3)

    # Final embedding block
    c4 = Conv2D(256, (4,4), activation='relu')(m3)
    f1 = Flatten()(c4)
    d1 = Dense(4096, activation='sigmoid')(f1)

    return Model(inputs=inp, outputs=d1, name='embedding')


In [ ]:
# Instantiate the embedding network and inspect its architecture.
embedding = make_embedding()
embedding.summary()


In [ ]:
# Custom distance layer for comparing two image embeddings.
class L1Dist(Layer):
    def __init__(self, **kwargs):
        super().__init__()

    def call(self, input_embedding, validation_embedding):
        """Return the element-wise absolute distance between two embeddings."""
        return tf.math.abs(input_embedding - validation_embedding)


In [ ]:
# Assemble the full Siamese network: two inputs, shared embedding model,
# distance calculation, then a sigmoid classifier for match / non-match.
def make_siamese_model():

    input_image = Input(name='input_img', shape=(100, 100, 3))
    validation_image = Input(name='validation_img', shape=(100, 100, 3))

    siamese_layer = L1Dist()
    siamese_layer._name = 'distance_layer'
    distances = siamese_layer(embedding(input_image), embedding(validation_image))

    classifier = Dense(1, activation='sigmoid')(distances)

    return Model(inputs=[input_image, validation_image], outputs=classifier, name='SiameseNetwork')


In [ ]:
# Instantiate the complete Siamese model and inspect its architecture.
siamese_model = make_siamese_model()
siamese_model.summary()


## 6. Training Configuration

The model is trained using **binary cross-entropy** and the **Adam optimiser**.  
Checkpointing is included so that progress can be saved periodically during longer runs.


In [ ]:
# Configure the training objective and optimiser.
binary_cross_loss = tf.losses.BinaryCrossentropy()
opt = tf.keras.optimizers.Adam(1e-4)


In [ ]:
# Store generated checkpoints and models outside the source tree.
artifact_dir = str(ARTIFACTS_DIR / 'face_verification')
checkpoint_dir = os.path.join(artifact_dir, 'checkpoints')
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(opt=opt, siamese_model=siamese_model)


In [ ]:
@tf.function
def train_step(batch):
    """Run a single optimisation step on one batch of training data."""
    with tf.GradientTape() as tape:
        # Separate the paired inputs from the ground-truth label.
        X = batch[:2]
        y = batch[2]

        # Forward pass through the Siamese network.
        yhat = siamese_model(X, training=True)

        # Compute binary cross-entropy loss.
        loss = binary_cross_loss(y, yhat)

    # Backpropagate and update trainable weights.
    grad = tape.gradient(loss, siamese_model.trainable_variables)

    opt.apply_gradients(zip(grad, siamese_model.trainable_variables))

    return loss


In [ ]:
def train(data, EPOCHS):
    """Train the model for a fixed number of epochs and save checkpoints."""
    for epoch in range(1, EPOCHS+1):
        print(f'Epoch {epoch}/{EPOCHS}')
        progbar = tf.keras.utils.Progbar(len(data))

        for idx, batch in enumerate(data):
            loss = train_step(batch)
            progbar.update(idx+1)

        if epoch % 10 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)


In [ ]:
# Launch training.
EPOCHS = 50
train(train_data, EPOCHS)


## 7. Evaluation

The prototype evaluation cell:
- samples one batch from the shuffled pair-level test partition
- generates predictions
- thresholds the predictions into match / non-match decisions
- calculates **precision** and **recall**
- visualises one example pair

This is only a pipeline smoke test. It is not identity- or session-disjoint, it evaluates one batch, and this notebook contains no saved metric output; it therefore does not support a deployment or accuracy claim.


In [ ]:
# Metrics used to evaluate verification performance on the held-out test split.
from tensorflow.keras.metrics import Precision, Recall


In [ ]:
# Pull one test batch for quick qualitative and quantitative evaluation.
test_input, test_val, y_true = test_data.as_numpy_iterator().next()


In [ ]:
# Generate predictions for the sampled test batch.
y_hat = siamese_model.predict([test_input, test_val])
y_hat


In [ ]:
# Convert probabilities into binary predictions using a 0.5 decision threshold.
y_hat = [1 if prediction > 0.5 else 0 for prediction in y_hat]


In [ ]:
# Ground-truth labels for the sampled batch.
y_true


In [ ]:
# Calculate precision and recall for the sampled predictions.
precision = Precision()
recall = Recall()

# Update the metric objects with the true labels and predicted labels.
precision.update_state(y_true, y_hat)
recall.update_state(y_true, y_hat)
print("Precision:", precision.result().numpy())
print("Recall:", recall.result().numpy())


In [ ]:
# Visualise one example pair from the test batch.
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(test_input[0])
plt.title("Input Image")

plt.subplot(1, 2, 2)
plt.imshow(test_val[0])
plt.title("Validation Image")


## 8. Saving and Reloading the Model

After training, the model is saved to disk and reloaded with the custom distance layer registered.  
This step is important because the verification application uses the reloaded model rather than the in-memory training object.


In [ ]:
# Persist the trained model to the ignored artifact directory.
model_path = os.path.join(artifact_dir, 'siamese_model.keras')
siamese_model.save(model_path)


In [ ]:
# Reload the saved model, including the custom distance layer.
model = tf.keras.models.load_model(model_path, custom_objects={'L1Dist': L1Dist})


## 9. Preparing the Verification Application

Before running real-time verification, the notebook prepares two folders:
- **verification**: a bank of stored reference images
- **input_verification**: the latest live image captured from the webcam

The verification function compares the input image against all stored reference images and uses two thresholds:
- **detection threshold**: when an individual comparison counts as a match
- **verification threshold**: how many matches are required overall to verify identity


In [ ]:
# Copy 50 positive images into the verification folder.
# The original training captures remain intact.
VERI_PATH = str(FACE_DATA_DIR / 'application_data' / 'verification')
if not os.path.exists(VERI_PATH):
    os.makedirs(VERI_PATH)

positive_images = os.listdir(POS_PATH)
for img in positive_images[:50]:
    img_path = os.path.join(POS_PATH, img)
    new_img_path = os.path.join(VERI_PATH, img)
    if not os.path.exists(new_img_path):
        shutil.copy2(img_path, new_img_path)


In [ ]:
# Create the folder that stores the live input frame captured during verification.
INPUT_VERI_PATH = str(FACE_DATA_DIR / 'application_data' / 'input_verification')
if not os.path.exists(INPUT_VERI_PATH):
    os.makedirs(INPUT_VERI_PATH)


In [ ]:
def verify(model, detection_threshold, verification_threshold):
    """Compare a live input image against the stored verification images."""
    # Build results array
    results = []
    for img in os.listdir(VERI_PATH):
        input_img = preprocess(os.path.join(INPUT_VERI_PATH, 'input_image.jpg'))
        validation_img = preprocess(os.path.join(VERI_PATH, img))

        # Make predictions for each stored verification image.
        result = model.predict(list(np.expand_dims([input_img, validation_img], axis=1)))
        results.append(result)

    # detection_threshold: score above which an individual comparison counts as a match
    detection = np.sum(np.array(results) > detection_threshold)

    # verification_threshold: proportion of matches required to verify identity
    verification = detection / len(os.listdir(VERI_PATH))
    verified = verification > verification_threshold

    return results, verified


In [ ]:
# Open the webcam feed and trigger verification in real time.
# Controls:
# - Press 'v' to run verification
# - Press 'q' to quit
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Unable to read a frame from the webcam.")
        break

    # Apply the same resolution-independent crop used during collection.
    frame = center_square_crop(frame)

    # Display the webcam feed back to the user, flipped like a mirror.
    cv2.imshow('Verification', cv2.flip(frame, 1))

    # Save the current frame and run the verification pipeline.
    if cv2.waitKey(1) & 0xFF == ord('v'):
        cv2.imwrite(os.path.join(INPUT_VERI_PATH, "input_image.jpg"), frame)
        results, verified = verify(model, detection_threshold=0.5, verification_threshold=0.2)

        if verified:
            print("User Verified")
        else:
            print("User Unverified")

    # Exit the loop cleanly when the user presses 'q'.
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the webcam and close any OpenCV windows.
cap.release()
cv2.destroyAllWindows()


## Results and takeaways

The notebook implements the complete experimental path from enrollment captures and pair construction through Siamese training, model persistence, thresholded matching, and webcam inference. Because every code cell is intentionally unexecuted in the repository, there is **no saved precision, recall, false-accept rate, or false-reject rate to report**. The work demonstrates system integration, not validated biometric performance.

## Limitations and next steps

This prototype connects dataset preparation, a custom Siamese architecture, TensorFlow training, and OpenCV inference. It is intentionally local and is not suitable for security decisions in its current form.

### Required improvements before broader use

- create identity- and session-disjoint train, validation, and test splits
- evaluate the full held-out set with ROC, false-accept, and false-reject rates
- calibrate thresholds on validation data instead of selecting them ad hoc
- test robustness across lighting, pose, cameras, and demographic groups
- minimize data retention and document consent, access, deletion, and threat models
- reduce model size and package inference only after the evaluation is complete
